# QSS 45 Final Project — Models
**Author:** Balla Sy

This notebook contains only the modeling portion of the QSS 45 final project.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Find the cleaned dataset whether this notebook is run
# from the project root or from inside code/.
cwd = Path.cwd()

possible_data_paths = [
    cwd / "qss45_bronx_manhattan_clean.csv",
    cwd / "data" / "qss45_bronx_manhattan_clean.csv",
    cwd.parent / "data" / "qss45_bronx_manhattan_clean.csv"
]

DATA_PATH = next(
    (path for path in possible_data_paths if path.exists()),
    possible_data_paths[-1]
)

# Save outputs to output/ when possible.
if cwd.name == "code":
    OUTPUT_DIR = cwd.parent / "output"
else:
    OUTPUT_DIR = cwd / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Data path:", DATA_PATH)
print("Output folder:", OUTPUT_DIR)


## 1. Load cleaned data

In [ ]:
data = pd.read_csv(DATA_PATH)

data["Bronx"] = (data["Borough"] == "Bronx").astype(int)

print(data.shape)
data.head()


## 2. Define predictors and outcome

In [ ]:
features = [
    "Transit",
    "Income",
    "Poverty",
    "Unemployment",
    "WorkAtHome",
    "Bronx"
]

target = "MeanCommute"

X = data[features]
y = data[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 3. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=45,
    stratify=data["Borough"]
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 4. Predictive OLS model

In [ ]:
ols_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])

ols_model.fit(X_train, y_train)

ols_predictions = ols_model.predict(X_test)

ols_r2 = r2_score(y_test, ols_predictions)
ols_rmse = mean_squared_error(y_test, ols_predictions) ** 0.5

print("OLS Test R²:", round(ols_r2, 3))
print("OLS Test RMSE:", round(ols_rmse, 3))


In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    y_test,
    ols_predictions,
    alpha=0.6
)

minimum = min(y_test.min(), ols_predictions.min())
maximum = max(y_test.max(), ols_predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Mean Commute Time (Minutes)")
plt.ylabel("Predicted Mean Commute Time (Minutes)")
plt.title("OLS: Actual vs. Predicted Commute Time")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ols_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 5. Model 1 — Borough only

In [ ]:
X_bronx_only = sm.add_constant(data[["Bronx"]])

model1 = sm.OLS(
    y,
    X_bronx_only
).fit(cov_type="HC3")

model1.summary()


## 6. Model 2 — Full controls

In [ ]:
X_full = data[
    [
        "Bronx",
        "Transit",
        "Income",
        "Poverty",
        "Unemployment",
        "WorkAtHome"
    ]
]

X_full = sm.add_constant(X_full)

model2 = sm.OLS(
    y,
    X_full
).fit(cov_type="HC3")

model2.summary()


## 7. Compare Model 1 and Model 2

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Model 1: Borough only",
        "Model 2: Full controls"
    ],
    "Bronx coefficient": [
        model1.params["Bronx"],
        model2.params["Bronx"]
    ],
    "R_squared": [
        model1.rsquared,
        model2.rsquared
    ],
    "Adjusted_R_squared": [
        model1.rsquared_adj,
        model2.rsquared_adj
    ]
})

comparison.round(3)


In [ ]:
gap_reduction = (
    (model1.params["Bronx"] - model2.params["Bronx"])
    / model1.params["Bronx"]
) * 100

print(
    "Percent reduction in Bronx coefficient:",
    round(gap_reduction, 1),
    "%"
)


## 8. Full-model coefficient table

In [ ]:
model2_results = pd.DataFrame({
    "Variable": model2.params.index,
    "Coefficient": model2.params.values,
    "Std_Error": model2.bse.values,
    "P_Value": model2.pvalues.values,
    "CI_Low": model2.conf_int()[0].values,
    "CI_High": model2.conf_int()[1].values
})

model2_results.round(4)


## 9. Full-model coefficient plot

In [ ]:
plot_data = (
    model2_results[
        model2_results["Variable"] != "const"
    ]
    .copy()
    .sort_values("Coefficient")
)

errors = np.vstack([
    plot_data["Coefficient"] - plot_data["CI_Low"],
    plot_data["CI_High"] - plot_data["Coefficient"]
])

plt.figure(figsize=(8, 5))

plt.errorbar(
    plot_data["Coefficient"],
    plot_data["Variable"],
    xerr=errors,
    fmt="o",
    capsize=4
)

plt.axvline(0, linestyle="--", linewidth=1)

plt.xlabel("OLS Coefficient")
plt.ylabel("")
plt.title("Factors Associated with Mean Commute Time")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_ols_coefficients.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 10. Residual diagnostic

In [ ]:
fitted = model2.fittedvalues
residuals = model2.resid

plt.figure(figsize=(7, 5))

plt.scatter(
    fitted,
    residuals,
    alpha=0.6
)

plt.axhline(0, linestyle="--")

plt.xlabel("Fitted Mean Commute Time")
plt.ylabel("Residuals")
plt.title("OLS Residuals vs. Fitted Values")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "ols_residual_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 11. Multicollinearity diagnostic

In [ ]:
X_vif = data[
    [
        "Bronx",
        "Transit",
        "Income",
        "Poverty",
        "Unemployment",
        "WorkAtHome"
    ]
].copy()

X_vif = sm.add_constant(X_vif)

vif_table = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif_table.round(2)


## 12. Bronx gap comparison figure

In [ ]:
bronx_compare = pd.DataFrame({
    "Model": [
        "Borough only",
        "Full controls"
    ],
    "Coefficient": [
        model1.params["Bronx"],
        model2.params["Bronx"]
    ],
    "CI_Low": [
        model1.conf_int().loc["Bronx", 0],
        model2.conf_int().loc["Bronx", 0]
    ],
    "CI_High": [
        model1.conf_int().loc["Bronx", 1],
        model2.conf_int().loc["Bronx", 1]
    ]
})

errors = np.vstack([
    bronx_compare["Coefficient"] - bronx_compare["CI_Low"],
    bronx_compare["CI_High"] - bronx_compare["Coefficient"]
])

plt.figure(figsize=(7, 5))

plt.errorbar(
    bronx_compare["Model"],
    bronx_compare["Coefficient"],
    yerr=errors,
    fmt="o",
    capsize=5
)

plt.ylabel("Estimated Bronx Commute Gap (Minutes)")
plt.title("Bronx Commute Gap Before and After Controls")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "final_bronx_gap_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 13. Save model outputs

In [ ]:
comparison.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index=False
)

vif_table.to_csv(
    OUTPUT_DIR / "vif_table.csv",
    index=False
)

model2_results.to_csv(
    OUTPUT_DIR / "full_ols_results.csv",
    index=False
)

print("Model outputs saved.")


## 14. Final model summary

In [ ]:
print("QSS 45 OLS Model Summary")
print("------------------------")

print("Raw Bronx commute gap:",
      round(model1.params["Bronx"], 2),
      "minutes")

print("Adjusted Bronx commute gap:",
      round(model2.params["Bronx"], 2),
      "minutes")

print("Model 1 R²:",
      round(model1.rsquared, 3))

print("Model 2 R²:",
      round(model2.rsquared, 3))

print("Predictive OLS Test R²:",
      round(ols_r2, 3))

print("Predictive OLS Test RMSE:",
      round(ols_rmse, 3),
      "minutes")

print("Percent reduction in Bronx coefficient:",
      round(gap_reduction, 1),
      "%")
